# 판정 인코더 v8 — 후보 두 개를 같은 행으로 견준다 (예전 dev 판 A ↔ 실제 문구 dev 판 R0) (박수진, 2026-10-06)

> 🔴 **test 는 건드리지 않는다** (D-175). 이 노트북은 학습 분할 안의 dev 행만 쓴다. 학습도 하지 않는다 — Drive 에 저장된 모델 두 개를 불러 확률만 낸다.

## 무엇을 견주나

| | A — 예전 dev 판 (v8.2 A) | R0 — 실제 문구 dev 판 (v8.3) |
|---|---|---|
| 모델을 고른 dev | 463행 (합성 포함 · 6종 층화) | 341행 (실제 문구만 · 8종 층화) |
| 저장 zip | `…-낱말제외-확정8.zip` | `…-낱말제외-확정8-dev실제-dev새2종.zip` |
| 학습 설정 | 같다 (유형 판 · 낱말 제외 · 9칸) | 같다 |

두 모델은 서로 다른 dev 로 쟀기 때문에 수치를 바로 견줄 수 없었다. 여기서는 **두 dev 에 모두 들어 있는 행**만 골라 두 모델을 같이 잰다.

- 그 행들은 어느 모델도 학습하지 않았다(각 모델은 자기 dev 묶음을 학습에서 뺐다) → 공정하다.
- R0 의 dev 에는 합성 행이 없으므로 공통 행은 전부 **실제 문구**다.

## 읽는 법 (돌리기 전에 정한다)

1. **PR-AUC** 가 모델 자체의 차이다(문턱과 무관). 6종 macro PR-AUC 의 차이(R0 − A)와 그 95% 구간(행을 다시 뽑는 부트스트랩)을 찍는다.
   - 구간이 0 을 포함하면 **「차이를 말할 수 없다」** 로 읽는다.
2. **재현율 · 정밀도 · 음성 오판정**은 각 모델이 자기 dev 에서 고른 문턱으로 잰다 — 문턱의 차이가 여기 나온다.
3. 새 2종은 각자의 문턱과 0.5 를 둘 다 찍는다(A 의 문턱은 0.900 · 0.800 이다).
4. 모델 하나씩(각 실행의 선택 시드)을 견주는 것이다. 시드 평균이 아니므로 작은 차이는 시드 잡음일 수 있다.

## 실행

1. 섹션 1 에서 `golden.jsonl` 을 올린다(사전 파일은 필요 없다).
2. Drive 에 두 zip 이 있어야 한다. 이름이 다르면 섹션 1 의 `ZIP_A` · `ZIP_R0` 를 고친다.
3. 위에서 아래로 한 번 실행한다. 1 ~ 2분이면 끝난다.


## 1. 파일 · 저장 모델 확인

In [ ]:
import os, json, hashlib, math, random, shutil, zipfile
from collections import Counter, defaultdict
import numpy as np

from google.colab import files as colab_files, drive
drive.mount("/content/drive", force_remount=False)
DRIVE = "/content/drive/MyDrive/"
ZIP_A = "copylane-encoder-kcbert-v8-all-낱말제외-확정8.zip"                      # 예전 dev 판 (v8.2 A)
ZIP_R0 = "copylane-encoder-kcbert-v8-all-낱말제외-확정8-dev실제-dev새2종.zip"     # 실제 문구 dev 판 (v8.3 R0)
for _z in (ZIP_A, ZIP_R0):
    if not os.path.exists(DRIVE + _z):
        print("Drive 의 v8 zip:", sorted(f for f in os.listdir(DRIVE) if f.startswith("copylane-encoder-kcbert-v8")))
        raise SystemExit(f"🔴 Drive 에 {_z} 가 없다 — 위 목록에서 이름을 찾아 ZIP_A · ZIP_R0 를 고친다")

if not os.path.exists("golden.jsonl"):
    print("golden.jsonl 선택")
    colab_files.upload()
GOLDEN_SHA = hashlib.sha256(open("golden.jsonl", "rb").read()).hexdigest()
EXPECTED_GOLDEN_SHA = "529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe"   # 재동결 10-05 (8,383행)
print("golden sha256:", GOLDEN_SHA[:16])
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176)")

def load_scheme(zname, out):
    shutil.rmtree(out, ignore_errors=True)
    zipfile.ZipFile(DRIVE + zname).extractall(out)
    return json.load(open(f"{out}/label_scheme.json", encoding="utf-8"))
SCH = {"A": load_scheme(ZIP_A, "./model_A"), "R0": load_scheme(ZIP_R0, "./model_R0")}
for k, s in SCH.items():
    assert s["golden_sha256"] == GOLDEN_SHA, f"{k}: 다른 골든으로 학습한 모델이다 ({s['golden_sha256'][:12]})"
    assert s.get("use_word_rows") is False and s.get("label_unit", "type") == "type", f"{k}: 낱말 제외 · 유형 판이 아니다"
    print(f"{k:3s} {s['notebook']} · 시드 {s['seed']} · {s['experiment']}")
assert SCH["A"]["label_list"] == SCH["R0"]["label_list"] and len(SCH["A"]["label_list"]) == 9, "두 모델의 출력 칸이 다르다"
assert "dev_sha" not in SCH["A"], "ZIP_A 가 예전 dev 판이 아니다 (dev 지문이 있다 — v8.3 이후 저장판)"
assert SCH["R0"].get("dev_definition", "").startswith("실제 문구만"), "ZIP_R0 가 실제 문구 dev 판이 아니다"


## 2. dev 두 개를 다시 만들고 공통 행을 고른다

In [ ]:
# ── dev 두 개를 다시 만든다 — 학습 노트북(v8.2 · v8.3) 섹션 2 · 4 와 같은 규칙. 저장 모델에 적힌 값과 대조해 같은 행인지 확인한다
CONFIRMED = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]
NEW_CONF = ["부당_비교광고", "비방광고"]
WAITING = ["기능성화장품_오인"]
CONF8 = CONFIRMED + NEW_CONF
LABEL_LIST = CONF8 + WAITING
assert SCH["A"]["label_list"] == LABEL_LIST
DEV_FRAC, DEV_SEED = 0.15, 42

def group_key(rid):
    return rid.split(":", 2)[2] if rid.startswith("inj:") else rid.split("#")[0]
def is_inj(r):
    return r.get("origin") == "injected" or str(r["id"]).startswith("inj:")
def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"
def train_role(r):
    c = r["condition"]
    if c == "M":
        return "drop_M"
    if c in ("D", "L"):
        return "neg"
    if r["labels"]:
        return "pos"
    return "pos_untyped" if c in ("C", "A", "B") else "neg"

golden = [json.loads(l) for l in open("golden.jsonl", encoding="utf-8") if l.strip()]
train_use = []
for g in golden:
    if g["split"] != "train":
        continue          # 🔴 평가 분할은 읽지 않는다 (D-175)
    r = {"id": g["id"], "group": group_key(g["id"]), "text": g["text"], "labels": g.get("labels") or [], "provenance": g.get("provenance"),
         "origin": g.get("origin"), "zone": g.get("구역"), "condition": g.get("조건"), "unit": g.get("unit")}
    role = train_role(r)
    if role != "drop_M":
        train_use.append(dict(r, role=role, labels=list(r["labels"]) if role == "pos" else []))

def make_dev(score_injected, stratify_new):
    def devable(r):
        return (not is_reason(r) and r["unit"] != "낱말" and r["role"] != "pos_untyped" and (score_injected or not is_inj(r)))
    groups = defaultdict(list)
    for r in train_use:
        groups[r["group"]].append(r)
    gkeys = sorted(groups)
    random.Random(DEV_SEED).shuffle(gkeys)
    strat = CONFIRMED + (NEW_CONF if stratify_new else [])
    pos_total = Counter(l for r in train_use if devable(r) for l in r["labels"] if l in strat)
    target = {l: max(1, math.ceil(DEV_FRAC * pos_total[l])) for l in strat}
    neg_groups = [g for g in gkeys if any(devable(r) and not r["labels"] for r in groups[g])]
    neg_target = math.ceil(DEV_FRAC * len(neg_groups))
    neg_set, got, neg_got, dev_groups = set(neg_groups), Counter(), 0, set()
    for g in gkeys:
        labs = Counter(l for r in groups[g] if devable(r) for l in r["labels"] if l in strat)
        need = any(got[l] < target[l] for l in labs)
        is_neg = g in neg_set and not labs
        if need or (is_neg and neg_got < neg_target):
            dev_groups.add(g); got.update(labs); neg_got += is_neg
    return [r for g in dev_groups for r in groups[g] if devable(r)], dev_groups

dev_A, groups_A = make_dev(score_injected=True, stratify_new=False)     # 예전 dev — 합성 포함 · 6종 층화
dev_R0, groups_R0 = make_dev(score_injected=False, stratify_new=True)   # 실제 문구 dev — 8종 층화
sha_R0 = hashlib.sha256("|".join(sorted(r["id"] for r in dev_R0)).encode()).hexdigest()[:16]
_npos = lambda rows: sum(1 for r in rows if any(l in CONFIRMED for l in r["labels"]))
print(f"예전 dev {len(dev_A)}행 (6종 양성 {_npos(dev_A)}) · 저장 모델 A 에 적힌 값 {SCH['A']['dev8']['n_dev']}행 (양성 {SCH['A']['dev_cond']['n_pos']})")
print(f"실제 문구 dev {len(dev_R0)}행 · 지문 {sha_R0} · 저장 모델 R0 에 적힌 값 {SCH['R0']['dev_rows']}행 · 지문 {SCH['R0']['dev_sha']}")
assert len(dev_A) == SCH["A"]["dev8"]["n_dev"] and _npos(dev_A) == SCH["A"]["dev_cond"]["n_pos"], "예전 dev 가 모델 A 를 고른 dev 와 다르다"
assert sha_R0 == SCH["R0"]["dev_sha"] and len(dev_R0) == SCH["R0"]["dev_rows"], "실제 문구 dev 가 모델 R0 을 고른 dev 와 다르다"

# 공통 행 — 두 dev 에 모두 든 행. 어느 모델도 학습하지 않았다(묶음째 학습에서 빠졌다)
ids_A = {r["id"] for r in dev_A}
common = [r for r in dev_R0 if r["id"] in ids_A]
assert common and all(r["group"] in groups_A and r["group"] in groups_R0 for r in common) and not any(is_inj(r) for r in common)
Yc = np.array([[int(l in r["labels"]) for l in LABEL_LIST] for r in common])
neg = np.array([not r["labels"] for r in common])
cond = np.array([r["condition"] or "없음" for r in common])
print(f"\n공통 행 {len(common)} (예전 dev 의 실제 문구 {sum(1 for r in dev_A if not is_inj(r))}행 · 실제 문구 dev {len(dev_R0)}행 중)")
print("  6종 양성 행", int(Yc[:, :6].any(1).sum()), "· 음성", int(neg.sum()), dict(Counter(map(str, cond[neg]))), "· 새 2종만 있는 행", int((~Yc[:, :6].any(1) & ~neg).sum()))
print("  유형별 양성:", {l: int(Yc[:, i].sum()) for i, l in enumerate(CONF8)})


## 3. 두 모델의 확률

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def predict(model_dir, rows, bs=64):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).eval()
    assert [mdl.config.id2label[i] for i in range(mdl.config.num_labels)] == LABEL_LIST, "모델의 칸 순서가 다르다"
    if torch.cuda.is_available():
        mdl = mdl.cuda()
    out = []
    for s in range(0, len(rows), bs):
        enc = tok([r["text"] for r in rows[s:s + bs]], truncation=True, max_length=128, padding=True, return_tensors="pt").to(mdl.device)
        with torch.no_grad():
            out.append(torch.sigmoid(mdl(**enc).logits.float()).cpu().numpy())
    return np.concatenate(out)

P = {"A": predict("./model_A", common), "R0": predict("./model_R0", common)}
TH = {k: np.array([SCH[k]["label_thresholds"][l] for l in CONF8]) for k in P}
print("문턱 (각 모델이 자기 dev 에서 고른 값)")
print(f"{'유형':20s} {'A':>7s} {'R0':>7s}")
for i, l in enumerate(CONF8):
    print(f"{l:20s} {TH['A'][i]:7.3f} {TH['R0'][i]:7.3f}")


## 4. 비교

In [ ]:
from sklearn.metrics import average_precision_score

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))
def ap(y, p):
    return float(average_precision_score(y, p)) if 0 < y.sum() < len(y) else None
def macro6(y, p):
    v = [ap(y[:, i], p[:, i]) for i in range(6)]
    v = [x for x in v if x is not None]
    return float(np.mean(v)) if v else float("nan")
fmt = lambda v: "  N/A" if v is None else f"{v:.3f}"

# ── 1) 모델 자체의 차이 — PR-AUC (문턱과 무관)
print(f"=== 1) PR-AUC — 공통 행 {len(common)} (전부 실제 문구 · 두 모델 모두 학습하지 않은 행) ===")
print(f"{'유형':20s} {'양성':>5s} {'A':>7s} {'R0':>7s} {'R0−A':>8s}")
for i, l in enumerate(CONF8):
    a, b = ap(Yc[:, i], P["A"][:, i]), ap(Yc[:, i], P["R0"][:, i])
    n = int(Yc[:, i].sum())
    d = "" if a is None or b is None else f"{b - a:+8.3f}"
    print(f"{l:20s} {n:5d} {fmt(a):>7s} {fmt(b):>7s} {d:>8s}" + ("  측정 불가(n<30)" if n < 30 else "") + ("  🆕" if l in NEW_CONF else ""))
mA, mR = macro6(Yc, P["A"]), macro6(Yc, P["R0"])
rs, diffs = np.random.RandomState(0), []
for _ in range(2000):   # 행을 다시 뽑는다(같은 행으로 두 모델을 잰다 — 짝 지은 부트스트랩)
    idx = rs.randint(0, len(common), len(common))
    diffs.append(macro6(Yc[idx], P["R0"][idx]) - macro6(Yc[idx], P["A"][idx]))
lo, hi = np.nanpercentile(diffs, [2.5, 97.5])
print(f"\n6종 macro PR-AUC   A {mA:.3f} · R0 {mR:.3f} · 차이(R0 − A) {mR - mA:+.3f}  95% 구간 [{lo:+.3f}, {hi:+.3f}]")
print("  →", "구간이 0 을 포함한다 — 이 행들로는 두 모델의 차이를 말할 수 없다" if lo <= 0 <= hi else
      ("R0 가 높다 (구간이 0 위)" if lo > 0 else "A 가 높다 (구간이 0 아래)"))

# ── 2) 문턱의 차이 — 각 모델이 자기 dev 에서 고른 문턱으로
pred = {k: (P[k][:, :8] >= TH[k]) for k in P}
print(f"\n=== 2) 유형별 재현율 · 정밀도 — 각자의 문턱 (Wilson 95%) ===")
print(f"{'유형':20s} {'양성':>5s}   {'A 재현율':>22s} {'A 정밀도':>8s}   {'R0 재현율':>22s} {'R0 정밀도':>8s}")
for i, l in enumerate(CONF8):
    n = int(Yc[:, i].sum()); row = f"{l:20s} {n:5d}   "
    for k in ("A", "R0"):
        tp = int((pred[k][:, i] & (Yc[:, i] == 1)).sum()); pp = int(pred[k][:, i].sum())
        rl, rh = wilson(tp, n)
        row += (f"{tp:3d}/{n:<3d} {tp / max(n, 1) * 100:5.1f}% [{rl * 100:2.0f}~{rh * 100:3.0f}]" if n else f"{'—':>22s}") + f" {tp / max(pp, 1) * 100:7.1f}%   "
    print(row + ("측정 불가(n<30)" if n < 30 else ""))

print(f"\n=== 3) 묶음 지표 — 각자의 문턱 ===")
pos6, pos8 = Yc[:, :6].any(1), Yc[:, :8].any(1)
def line(name, fa, fr, n):
    print(f"{name:34s} A {fa:4d}/{n:<4d} {fa / max(n, 1) * 100:5.1f}%   R0 {fr:4d}/{n:<4d} {fr / max(n, 1) * 100:5.1f}%")
line("위반 재현율 (6종 기준)", int(pred["A"][pos6, :6].any(1).sum()), int(pred["R0"][pos6, :6].any(1).sum()), int(pos6.sum()))
line("위반 재현율 (8종 기준)", int(pred["A"][pos8].any(1).sum()), int(pred["R0"][pos8].any(1).sum()), int(pos8.sum()))
line("음성 오판정 (6종 기준)", int(pred["A"][neg, :6].any(1).sum()), int(pred["R0"][neg, :6].any(1).sum()), int(neg.sum()))
line("음성 오판정 (8종 기준)", int(pred["A"][neg].any(1).sum()), int(pred["R0"][neg].any(1).sum()), int(neg.sum()))
for c in sorted(set(map(str, cond[neg]))):
    m = neg & (cond == c)
    line(f"  조건 {c} (8종 기준)", int(pred["A"][m].any(1).sum()), int(pred["R0"][m].any(1).sum()), int(m.sum()))

# ── 4) 새 2종 — 각자의 문턱과 0.5
print(f"\n=== 4) 새 2종 — 실제 문구에서 (각자의 문턱 / 0.5) ===")
for l in NEW_CONF:
    i = CONF8.index(l); y = Yc[:, i] == 1; n = int(y.sum())
    for k in ("A", "R0"):
        own = int((P[k][y, i] >= TH[k][i]).sum()); half = int((P[k][y, i] >= 0.5).sum())
        fp_own = int((P[k][~y, i] >= TH[k][i]).sum()); fp_half = int((P[k][~y, i] >= 0.5).sum())
        print(f"  {l:10s} {k:3s} 양성 {n:2d} · 문턱 {TH[k][i]:.3f} → 잡음 {own}/{n} · 잘못 울림 {fp_own}   |   문턱 0.5 → 잡음 {half}/{n} · 잘못 울림 {fp_half}"
              + f"   · 양성 확률 {sorted(round(float(v), 2) for v in P[k][y, i])}" * (n <= 15))
print("\n(모델 하나씩을 견준 것이다 — 각 실행의 선택 시드. 시드 평균이 아니라서 작은 차이는 시드 잡음일 수 있다)")


## 결과를 어떻게 쓰나

- 이 비교는 **어느 dev 정의로 갈지**를 정하는 재료다. 정하는 것은 팀장이다(D-321 결정 4).
- 1) 의 구간이 0 을 포함하면 두 모델의 실력 차이는 말할 수 없다 — 그러면 남는 차이는 **문턱**(2 ~ 4)이다.
- 새 2종에서 A 의 문턱(0.900 · 0.800)과 0.5 의 차이가 크면, 예전 dev 판으로 가더라도 새 2종 문턱을 어떻게 둘지 따로 정해야 한다.
- 🔴 이 결과로 문턱을 새로 고르지 않는다 — 공통 행은 두 dev 의 일부라 여기서 고른 값은 어느 실행의 규칙으로도 설명되지 않는다.
